In [1]:

import json
from pathlib import Path

# Locate the collected RabbitMQ batch.
batch_file = Path(
    r"C:\Data_Engineering\freshcart-pipelines"
    r"\bonus\bonus_orders_batch_001.jsonl"
)

# Read the saved orders.
with batch_file.open("r", encoding="utf-8") as f:
    orders = [json.loads(line) for line in f if line.strip()]

# Check required FreshCart fields.
required_fields = {
    "order_id",
    "cust_email",
    "cust_name",
    "order_date",
    "amount",
    "currency",
    "pincode",
    "updated_at",
}

invalid_orders = [
    order for order in orders
    if not required_fields.issubset(order)
]

order_ids = [order["order_id"] for order in orders]

print("Total orders:", len(orders))
print("Unique order IDs:", len(set(order_ids)))
print("Orders missing required fields:", len(invalid_orders))

assert len(orders) == 25, "Expected 25 collected orders"
assert len(set(order_ids)) == 25, "Duplicate order IDs found"
assert not invalid_orders, "Some orders are missing required fields"

print("Batch validation: PASSED")
print("\nFirst order:")
print(json.dumps(orders[0], indent=2))


Total orders: 25
Unique order IDs: 25
Orders missing required fields: 0
Batch validation: PASSED

First order:
{
  "order_id": "BONUS-ORDER-001",
  "cust_email": "priya.r@gmail.com",
  "cust_name": "Priya R",
  "order_date": "2026-09-03",
  "amount": "525",
  "currency": "INR",
  "pincode": "600041",
  "updated_at": "2026-09-03 12:00:00"
}


In [2]:

import gzip
import json
from datetime import datetime, timezone
from pathlib import Path

LANDING_DIR = Path(
    r"C:\Data_Engineering\freshcart-pipelines\bonus\landing"
)

RUN_ID = "BONUS-001"


def write_landing(source, records, run_id):
    folder = LANDING_DIR / source
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{source}__{run_id}.json.gz"
    extracted_at = datetime.now(timezone.utc).isoformat()

    with gzip.open(path, "wt", encoding="utf-8") as file:
        for record in records:
            envelope = {
                "source": source,
                "run_id": run_id,
                "extracted_at": extracted_at,
                "data": record,
            }
            file.write(json.dumps(envelope) + "\n")

    print(f"Landing file: {path}")
    print(f"Records written: {len(records)}")

    return path


landing_file = write_landing(
    source="web",
    records=orders,
    run_id=RUN_ID,
)

# Verify the compressed landing file.
with gzip.open(landing_file, "rt", encoding="utf-8") as file:
    landing_records = [
        json.loads(line)
        for line in file if line.strip()
    ]

assert len(landing_records) == 25
assert all(record["source"] == "web" for record in landing_records)

print("Landing file validation: PASSED")


Landing file: C:\Data_Engineering\freshcart-pipelines\bonus\landing\web\web__BONUS-001.json.gz
Records written: 25
Landing file validation: PASSED


In [4]:

from getpass import getpass
import snowflake.connector

# Use credentials from your original FreshCart notebook.
account = input("Snowflake account identifier: ").strip()
username = input("Snowflake username: ").strip()
password = getpass("Snowflake password: ")

conn = snowflake.connector.connect(
    account=account,
    user=username,
    password=password,
    warehouse="FRESHCART_WH",
    database="FRESHCART",
)

try:
    with conn.cursor() as cur:
        cur.execute("SELECT CURRENT_DATABASE(), CURRENT_WAREHOUSE()")
        print("Connected:", cur.fetchone())

        cur.execute("SELECT COUNT(*) FROM FRESHCART.BRONZE.ORDERS_RAW")
        print("Existing Bronze rows:", cur.fetchone()[0])

        cur.execute("SELECT COUNT(*) FROM FRESHCART.GOLD.FCT_ORDERS")
        print("Existing Gold rows:", cur.fetchone()[0])

        cur.execute("SHOW STAGES LIKE 'LANDING' IN SCHEMA FRESHCART.BRONZE")
        print("Landing stage found:", bool(cur.fetchall()))

except Exception as error:
    print("Connection or verification failed:", error)


Connected: ('FRESHCART', 'FRESHCART_WH')
Existing Bronze rows: 91
Existing Gold rows: 5
Landing stage found: True


In [5]:

# Read-only checks before loading bonus orders.

order_date = orders[0]["order_date"]
emails = sorted({o["cust_email"] for o in orders})
pincodes = sorted({o["pincode"] for o in orders})

with conn.cursor() as cur:
    print("CUSTOMER MATCHES")
    for email in emails:
        cur.execute("""
            SELECT cust_id, tier
            FROM FRESHCART.DIM.CUSTOMERS_SCD2
            WHERE LOWER(email) = %s
              AND %s::DATE BETWEEN valid_from AND valid_to
        """, (email, order_date))
        print(email, "->", cur.fetchall())

    print("\nPINCODE MATCHES")
    for pin in pincodes:
        cur.execute("""
            SELECT city, state
            FROM FRESHCART.REF.PINCODE_GEO
            WHERE pincode = %s
        """, (pin,))
        print(pin, "->", cur.fetchall())

    print("\nCURRENCY MATCH")
    cur.execute("""
        SELECT currency, rate_to_inr
        FROM FRESHCART.REF.FX_DAILY
        WHERE currency = 'INR'
          AND rate_date = %s::DATE
    """, (order_date,))
    print("INR ->", cur.fetchall())

    print("\nEXISTING BONUS ORDERS")
    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE order_id LIKE 'BONUS-ORDER-%'
    """)
    print("Bonus orders already in Gold:", cur.fetchone()[0])


CUSTOMER MATCHES
arjun.k@yahoo.in -> [('CUST-02', 'Gold')]
meera@outlook.com -> [('CUST-03', 'Platinum')]
priya.r@gmail.com -> [('CUST-01', 'Gold')]

PINCODE MATCHES
400050 -> [('Mumbai', 'Maharashtra')]
560034 -> [('Bengaluru', 'Karnataka')]
600041 -> [('Chennai', 'Tamil Nadu')]

CURRENCY MATCH
INR -> [('INR', Decimal('1.0000'))]

EXISTING BONUS ORDERS
Bonus orders already in Gold: 0


In [6]:

# Step 15: Load the RabbitMQ batch into Snowflake Bronze.

def load_file(local_path, table, stage_dir):
    uri = "file://" + Path(local_path).resolve().as_posix()

    with conn.cursor() as cur:
        cur.execute(
            f"PUT '{uri}' @bronze.landing/{stage_dir}/ "
            "AUTO_COMPRESS=FALSE OVERWRITE=FALSE"
        )
        print("PUT result:", cur.fetchall())

        cur.execute(f"""
            COPY INTO {table}
            (source, raw, run_id, extracted_at, src_file, src_row)
            FROM (
                SELECT
                    $1:source::STRING,
                    $1:data,
                    $1:run_id::STRING,
                    $1:extracted_at::TIMESTAMP_TZ,
                    METADATA$FILENAME,
                    METADATA$FILE_ROW_NUMBER
                FROM @bronze.landing/{stage_dir}/
            )
            FILE_FORMAT = (FORMAT_NAME = 'bronze.ff_ndjson')
            ON_ERROR = 'ABORT_STATEMENT'
        """)

        results = cur.fetchall()
        print("COPY results:", results)

        return results


# Check before loading.
with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.BRONZE.ORDERS_RAW
        WHERE run_id = 'BONUS-001'
    """)
    before = cur.fetchone()[0]

print("Bonus Bronze rows before:", before)

# Load once using a dedicated stage directory.
if before == 0:
    load_file(
        landing_file,
        "bronze.orders_raw",
        "bonus/orders_web/BONUS-001"
    )
else:
    print("Batch already present; skipping upload.")

# Verify loaded records.
with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*), COUNT(DISTINCT raw:order_id::STRING)
        FROM FRESHCART.BRONZE.ORDERS_RAW
        WHERE run_id = 'BONUS-001'
    """)
    total, unique = cur.fetchone()

print("Bonus Bronze rows:", total)
print("Unique bonus orders:", unique)

assert total == 25, "Unexpected Bronze row count"
assert unique == 25, "Duplicate order IDs detected"

print("BRONZE LOAD: PASSED")


Bonus Bronze rows before: 0
PUT result: [('web__BONUS-001.json.gz', 'web__BONUS-001.json.gz', 508, 512, 'GZIP', 'GZIP', 'UPLOADED', '')]
COPY results: [('landing/bonus/orders_web/BONUS-001/web__BONUS-001.json.gz', 'LOADED', 25, 25, 1, 0, None, None, None, None)]
Bonus Bronze rows: 25
Unique bonus orders: 25
BRONZE LOAD: PASSED


In [7]:

# Step 16: Verify the bonus records in Bronze.

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            raw:order_id::STRING AS order_id,
            raw:cust_email::STRING AS email,
            raw:amount::NUMBER(12,2) AS amount,
            raw:currency::STRING AS currency
        FROM FRESHCART.BRONZE.ORDERS_RAW
        WHERE run_id = 'BONUS-001'
        ORDER BY order_id
        LIMIT 5
    """)

    print("FIRST FIVE BONUS ORDERS")
    for row in cur.fetchall():
        print(row)

    cur.execute("""
        SELECT
            COUNT(*) AS total_orders,
            COUNT(DISTINCT raw:order_id::STRING)
                AS unique_orders,
            SUM(raw:amount::NUMBER(12,2))
                AS total_amount
        FROM FRESHCART.BRONZE.ORDERS_RAW
        WHERE run_id = 'BONUS-001'
    """)

    print("\nBONUS BATCH SUMMARY")
    print(cur.fetchone())


FIRST FIVE BONUS ORDERS
('BONUS-ORDER-001', 'priya.r@gmail.com', Decimal('525.00'), 'INR')
('BONUS-ORDER-002', 'arjun.k@yahoo.in', Decimal('550.00'), 'INR')
('BONUS-ORDER-003', 'meera@outlook.com', Decimal('575.00'), 'INR')
('BONUS-ORDER-004', 'priya.r@gmail.com', Decimal('600.00'), 'INR')
('BONUS-ORDER-005', 'arjun.k@yahoo.in', Decimal('625.00'), 'INR')

BONUS BATCH SUMMARY
(25, 25, Decimal('20625.00'))


In [8]:

# Step 17: Read-only checks before transformation.

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            COUNT(*) AS total,
            COUNT_IF(
                TRY_TO_DATE(raw:order_date::STRING) IS NULL
            ) AS bad_dates,
            COUNT_IF(
                TRY_TO_DECIMAL(raw:amount::STRING, 12, 2) <= 0
            ) AS bad_amounts,
            COUNT_IF(
                raw:currency::STRING NOT IN ('INR', 'USD')
            ) AS bad_currencies,
            COUNT_IF(
                raw:updated_at::STRING IS NULL
            ) AS missing_timestamps
        FROM FRESHCART.BRONZE.ORDERS_RAW
        WHERE run_id = 'BONUS-001'
    """)

    print("BONUS VALIDATION SUMMARY")
    print(cur.fetchone())

    cur.execute("""
        SELECT
            COUNT(*) AS total,
            COUNT(DISTINCT raw:order_id::STRING) AS unique_orders
        FROM FRESHCART.BRONZE.ORDERS_RAW
        WHERE run_id = 'BONUS-001'
    """)

    print("BONUS UNIQUENESS")
    print(cur.fetchone())

print("Read-only pre-transformation checks completed.")


BONUS VALIDATION SUMMARY
(25, 0, 0, 0, 0)
BONUS UNIQUENESS
(25, 25)
Read-only pre-transformation checks completed.


In [9]:

# Step 18: Inspect the existing table structures.
# READ ONLY - no data will be modified.

tables = [
    ("SILVER", "ORDERS_CHECKED"),
    ("SILVER", "ORDERS_CLEAN"),
    ("SILVER", "ORDERS_DEDUP"),
    ("SILVER", "ORDERS_ENRICHED"),
    ("GOLD", "FCT_ORDERS"),
]

with conn.cursor() as cur:
    for schema, table in tables:
        print(f"\n{schema}.{table}")

        cur.execute("""
            SELECT COLUMN_NAME, DATA_TYPE
            FROM FRESHCART.INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = %s
              AND TABLE_NAME = %s
            ORDER BY ORDINAL_POSITION
        """, (schema, table))

        columns = cur.fetchall()

        if not columns:
            print("Table not found")
        else:
            print(", ".join(
                name for name, dtype in columns
            ))

            cur.execute(
                f"SELECT COUNT(*) FROM "
                f"FRESHCART.{schema}.{table}"
            )
            print("Current rows:", cur.fetchone()[0])

print("\nTABLE INSPECTION COMPLETED")



SILVER.ORDERS_CHECKED
INGEST_SEQ, SOURCE, ORDER_ID, EMAIL, NAME, ORDER_DATE, AMOUNT, CURRENCY, PINCODE, UPDATED_AT, REASONS
Current rows: 91

SILVER.ORDERS_CLEAN
INGEST_SEQ, SOURCE, ORDER_ID, EMAIL, NAME, ORDER_DATE, AMOUNT, CURRENCY, PINCODE, UPDATED_AT, AMOUNT_MISSING
Current rows: 68

SILVER.ORDERS_DEDUP
INGEST_SEQ, SOURCE, ORDER_ID, EMAIL, NAME, ORDER_DATE, AMOUNT, CURRENCY, PINCODE, UPDATED_AT, AMOUNT_MISSING
Current rows: 5

SILVER.ORDERS_ENRICHED
INGEST_SEQ, SOURCE, ORDER_ID, EMAIL, NAME, ORDER_DATE, AMOUNT, CURRENCY, PINCODE, UPDATED_AT, AMOUNT_MISSING, CUST_ID, TIER, MULTIPLIER, FX_RATE, AMOUNT_INR, LOYALTY_POINTS, IS_HIGH_VALUE, CITY, STATE, REGION
Current rows: 5

GOLD.FCT_ORDERS
ORDER_ID, ORDER_DATE, CUST_ID, EMAIL, TIER, MULTIPLIER, AMOUNT, CURRENCY, FX_RATE, AMOUNT_INR, LOYALTY_POINTS, IS_HIGH_VALUE, CITY, STATE, REGION, UPDATED_AT, ROW_HASH, PUBLISHED_AT
Current rows: 5

TABLE INSPECTION COMPLETED


In [10]:

# STEP 19 — Transform bonus orders into Silver.
# Uses temporary tables; existing Silver and Gold data remain unchanged.

with conn.cursor() as cur:

    cur.execute("""
        CREATE OR REPLACE TEMP TABLE
            SILVER.BONUS_ORDERS_CLEAN AS
        SELECT
            ingest_seq,
            source,
            TRIM(raw:order_id::STRING) AS order_id,
            LOWER(TRIM(raw:cust_email::STRING)) AS email,
            INITCAP(TRIM(raw:cust_name::STRING)) AS name,
            TRY_TO_DATE(raw:order_date::STRING) AS order_date,
            TRY_TO_DECIMAL(
                raw:amount::STRING, 12, 2
            ) AS amount,
            UPPER(TRIM(raw:currency::STRING)) AS currency,
            TRIM(raw:pincode::STRING) AS pincode,
            TRY_TO_TIMESTAMP_NTZ(
                raw:updated_at::STRING
            ) AS updated_at,
            FALSE AS amount_missing
        FROM FRESHCART.BRONZE.ORDERS_RAW
        WHERE run_id = 'BONUS-001'
          AND source = 'web'
    """)

    cur.execute("""
        CREATE OR REPLACE TEMP TABLE
            SILVER.BONUS_ORDERS_DEDUP AS
        SELECT *
        FROM SILVER.BONUS_ORDERS_CLEAN
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY updated_at DESC, ingest_seq DESC
        ) = 1
    """)

    cur.execute("""
        SELECT COUNT(*)
        FROM SILVER.BONUS_ORDERS_CLEAN
    """)
    clean_count = cur.fetchone()[0]

    cur.execute("""
        SELECT COUNT(*), COUNT(DISTINCT order_id)
        FROM SILVER.BONUS_ORDERS_DEDUP
    """)
    dedup_count, unique_count = cur.fetchone()

print("Bonus Silver cleaned:", clean_count)
print("Bonus Silver deduplicated:", dedup_count)
print("Unique order IDs:", unique_count)

assert clean_count == 25
assert dedup_count == 25
assert unique_count == 25

print("SILVER TRANSFORMATION: PASSED")


Bonus Silver cleaned: 25
Bonus Silver deduplicated: 25
Unique order IDs: 25
SILVER TRANSFORMATION: PASSED


In [11]:

# STEP 20 — Enrich bonus orders.
# Uses the same logic as the original FreshCart ELT notebook.
# Existing Gold data remains unchanged.

with conn.cursor() as cur:

    cur.execute("""
        CREATE OR REPLACE TEMP TABLE
            SILVER.BONUS_ORDERS_ENRICHED AS
        WITH joined AS (
            SELECT
                o.*,
                c.cust_id,
                c.tier,
                m.multiplier,
                fx.rate_to_inr AS fx_rate,
                ROUND(
                    o.amount * fx.rate_to_inr, 2
                ) AS amount_inr,
                g.city,
                g.state,
                g.region
            FROM SILVER.BONUS_ORDERS_DEDUP o

            LEFT JOIN DIM.CUSTOMERS_SCD2 c
                ON c.email = o.email
                AND o.order_date
                    BETWEEN c.valid_from AND c.valid_to

            LEFT JOIN REF.TIER_MULTIPLIER m
                ON m.tier = c.tier

            LEFT JOIN REF.FX_DAILY fx
                ON fx.currency = o.currency
                AND fx.rate_date = o.order_date

            LEFT JOIN REF.PINCODE_GEO g
                ON g.pincode = o.pincode
        )

        SELECT
            *,
            FLOOR(amount_inr / 100)
                * multiplier AS loyalty_points,
            amount_inr >= 2000 AS is_high_value
        FROM joined
    """)

    cur.execute("""
        SELECT
            COUNT(*),
            COUNT(DISTINCT order_id),
            COUNT_IF(
                cust_id IS NULL
                OR multiplier IS NULL
                OR fx_rate IS NULL
                OR city IS NULL
                OR amount_inr IS NULL
                OR loyalty_points IS NULL
            ),
            SUM(amount_inr),
            SUM(loyalty_points)
        FROM SILVER.BONUS_ORDERS_ENRICHED
    """)

    total, unique, invalid, revenue, points = cur.fetchone()

print("BONUS ENRICHMENT RESULTS")
print("Enriched orders:", total)
print("Unique orders:", unique)
print("Missing enrichment values:", invalid)
print("Bonus revenue:", revenue)
print("Bonus loyalty points:", points)

assert total == 25
assert unique == 25
assert invalid == 0

print("BONUS ENRICHMENT: PASSED")


BONUS ENRICHMENT RESULTS
Enriched orders: 25
Unique orders: 25
Missing enrichment values: 0
Bonus revenue: 20625.00
Bonus loyalty points: 458
BONUS ENRICHMENT: PASSED


In [12]:

# STEP 21 — Safely merge bonus orders into Gold.
# Keep the notebook kernel and Snowflake connection active.

with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*), COALESCE(SUM(amount_inr), 0),
               COALESCE(SUM(loyalty_points), 0)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """)
    before_count, before_revenue, before_points = cur.fetchone()

    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS g
        JOIN SILVER.BONUS_ORDERS_ENRICHED b
          ON g.order_id = b.order_id
    """)
    existing_bonus = cur.fetchone()[0]

    print("Gold orders before:", before_count)
    print("Bonus order IDs already in Gold:", existing_bonus)

    assert existing_bonus == 0, (
        "Bonus orders already exist. Do not repeat the first merge."
    )

    cur.execute("""
        MERGE INTO FRESHCART.GOLD.FCT_ORDERS t
        USING (
            SELECT
                order_id, order_date, cust_id, email,
                tier, multiplier, amount, currency,
                fx_rate, amount_inr, loyalty_points,
                is_high_value, city, state, region,
                updated_at,
                HASH(
                    order_date, cust_id, email, tier,
                    multiplier, amount, currency,
                    fx_rate, amount_inr, loyalty_points,
                    is_high_value, city, state, region,
                    updated_at
                ) AS row_hash
            FROM SILVER.BONUS_ORDERS_ENRICHED
        ) s
        ON t.order_id = s.order_id

        WHEN MATCHED
          AND t.row_hash IS DISTINCT FROM s.row_hash
        THEN UPDATE SET
            order_date = s.order_date,
            cust_id = s.cust_id,
            email = s.email,
            tier = s.tier,
            multiplier = s.multiplier,
            amount = s.amount,
            currency = s.currency,
            fx_rate = s.fx_rate,
            amount_inr = s.amount_inr,
            loyalty_points = s.loyalty_points,
            is_high_value = s.is_high_value,
            city = s.city,
            state = s.state,
            region = s.region,
            updated_at = s.updated_at,
            row_hash = s.row_hash,
            published_at = CURRENT_TIMESTAMP()

        WHEN NOT MATCHED THEN INSERT (
            order_id, order_date, cust_id, email,
            tier, multiplier, amount, currency,
            fx_rate, amount_inr, loyalty_points,
            is_high_value, city, state, region,
            updated_at, row_hash, published_at
        )
        VALUES (
            s.order_id, s.order_date, s.cust_id,
            s.email, s.tier, s.multiplier, s.amount,
            s.currency, s.fx_rate, s.amount_inr,
            s.loyalty_points, s.is_high_value,
            s.city, s.state, s.region, s.updated_at,
            s.row_hash, CURRENT_TIMESTAMP()
        )
    """)

    print("MERGE result:", cur.fetchall())

    cur.execute("""
        SELECT COUNT(*), COALESCE(SUM(amount_inr), 0),
               COALESCE(SUM(loyalty_points), 0)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """)
    after_count, after_revenue, after_points = cur.fetchone()

    cur.execute("""
        SELECT COUNT(*), COALESCE(SUM(amount_inr), 0),
               COALESCE(SUM(loyalty_points), 0)
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE order_id LIKE 'BONUS-ORDER-%'
    """)
    bonus_count, bonus_revenue, bonus_points = cur.fetchone()

print("\nGOLD RESULTS")
print("Total Gold orders:", after_count)
print("Total Gold revenue:", after_revenue)
print("Total Gold loyalty points:", after_points)
print("Bonus orders in Gold:", bonus_count)
print("Bonus revenue:", bonus_revenue)
print("Bonus loyalty points:", bonus_points)

assert after_count == before_count + 25
assert after_revenue == before_revenue + bonus_revenue
assert after_points == before_points + bonus_points
assert bonus_count == 25
assert bonus_revenue == 20625
assert bonus_points == 458

print("GOLD MERGE: PASSED")


Gold orders before: 5
Bonus order IDs already in Gold: 0
MERGE result: [(25, 0)]

GOLD RESULTS
Total Gold orders: 30
Total Gold revenue: 36720.00
Total Gold loyalty points: 785
Bonus orders in Gold: 25
Bonus revenue: 20625.00
Bonus loyalty points: 458
GOLD MERGE: PASSED


In [13]:

# STEP 22 — Idempotency test.
# Re-executes the same MERGE without duplicating Gold orders.

with conn.cursor() as cur:

    # Record the state before rerunning.
    cur.execute("""
        SELECT
            COUNT(*),
            SUM(amount_inr),
            SUM(loyalty_points),
            MAX(published_at)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """)
    before = cur.fetchone()

    cur.execute("""
        SELECT order_id, published_at
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE order_id LIKE 'BONUS-ORDER-%'
        ORDER BY order_id
    """)
    published_before = dict(cur.fetchall())

    # Same source records and row_hash calculation as Step 21.
    cur.execute("""
        MERGE INTO FRESHCART.GOLD.FCT_ORDERS t
        USING (
            SELECT
                order_id, order_date, cust_id, email,
                tier, multiplier, amount, currency,
                fx_rate, amount_inr, loyalty_points,
                is_high_value, city, state, region,
                updated_at,
                HASH(
                    order_date, cust_id, email, tier,
                    multiplier, amount, currency,
                    fx_rate, amount_inr, loyalty_points,
                    is_high_value, city, state, region,
                    updated_at
                ) AS row_hash
            FROM SILVER.BONUS_ORDERS_ENRICHED
        ) s
        ON t.order_id = s.order_id

        WHEN MATCHED
          AND t.row_hash IS DISTINCT FROM s.row_hash
        THEN UPDATE SET
            order_date = s.order_date,
            cust_id = s.cust_id,
            email = s.email,
            tier = s.tier,
            multiplier = s.multiplier,
            amount = s.amount,
            currency = s.currency,
            fx_rate = s.fx_rate,
            amount_inr = s.amount_inr,
            loyalty_points = s.loyalty_points,
            is_high_value = s.is_high_value,
            city = s.city,
            state = s.state,
            region = s.region,
            updated_at = s.updated_at,
            row_hash = s.row_hash,
            published_at = CURRENT_TIMESTAMP()

        WHEN NOT MATCHED THEN INSERT (
            order_id, order_date, cust_id, email,
            tier, multiplier, amount, currency,
            fx_rate, amount_inr, loyalty_points,
            is_high_value, city, state, region,
            updated_at, row_hash, published_at
        )
        VALUES (
            s.order_id, s.order_date, s.cust_id,
            s.email, s.tier, s.multiplier, s.amount,
            s.currency, s.fx_rate, s.amount_inr,
            s.loyalty_points, s.is_high_value,
            s.city, s.state, s.region, s.updated_at,
            s.row_hash, CURRENT_TIMESTAMP()
        )
    """)

    merge_result = cur.fetchall()

    cur.execute("""
        SELECT
            COUNT(*),
            SUM(amount_inr),
            SUM(loyalty_points),
            MAX(published_at)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """)
    after = cur.fetchone()

    cur.execute("""
        SELECT order_id, published_at
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE order_id LIKE 'BONUS-ORDER-%'
        ORDER BY order_id
    """)
    published_after = dict(cur.fetchall())

print("SECOND MERGE RESULT:", merge_result)
print("\nBEFORE:", before)
print("AFTER: ", after)

assert before == after, "Gold totals changed after rerun!"
assert published_before == published_after, (
    "Published timestamps changed unexpectedly!"
)

print("\nIDEMPOTENCY TEST: PASSED")
print("No duplicate orders.")
print("Gold totals unchanged.")
print("Published timestamps unchanged.")


SECOND MERGE RESULT: [(0, 0)]

BEFORE: (30, Decimal('36720.00'), 785, datetime.datetime(2026, 11, 15, 9, 20, 31, 381000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
AFTER:  (30, Decimal('36720.00'), 785, datetime.datetime(2026, 11, 15, 9, 20, 31, 381000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))

IDEMPOTENCY TEST: PASSED
No duplicate orders.
Gold totals unchanged.
Published timestamps unchanged.
